# Digital Banking Clickstream & Customer Journey Analytics

**Portfolio project using synthetic data.**

Business objective: analyze digital-banking clickstream behavior, quantify funnel drop-off and conversion, test whether conversion differs across customer experience dimensions, and build a machine-learning model to predict journey completion.


## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


## 2. Load raw clickstream data


In [ ]:
df = pd.read_csv("../data/raw/digital_banking_clickstream.csv",
                 parse_dates=["event_timestamp", "session_start"])
print(df.shape)
df.head()


## 3. Data quality checks


In [ ]:
print(df.info())
print(df.isna().sum())
print("Duplicate event IDs:", df["event_id"].duplicated().sum())
print("Sessions:", df["session_id"].nunique())
print("Customers:", df["customer_id"].nunique())


## 4. Funnel analysis


In [ ]:
funnel = (df.groupby(["journey_step","event_name"])["session_id"]
          .nunique().reset_index(name="sessions")
          .sort_values("journey_step"))
funnel["overall_reach_pct"] = funnel["sessions"] / df["session_id"].nunique() * 100
funnel


In [ ]:
plt.figure(figsize=(9,5))
plt.bar(funnel["event_name"], funnel["sessions"])
plt.title("Digital Banking Journey Funnel")
plt.xlabel("Journey step")
plt.ylabel("Sessions")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


## 5. Session-level features


In [ ]:
session = (df.groupby("session_id")
           .agg(customer_id=("customer_id","first"),
                session_start=("session_start","first"),
                device_type=("device_type","first"),
                channel=("channel","first"),
                product=("product","first"),
                city=("city","first"),
                events_per_session=("event_id","count"),
                max_journey_step=("journey_step","max"),
                converted=("converted","max"),
                last_event_time=("event_timestamp","max"))
           .reset_index())
session["session_duration_seconds"] = (
    session["last_event_time"] - session["session_start"]
).dt.total_seconds()
session["day_of_week"] = session["session_start"].dt.day_name()
session.head()


## 6. KPI analysis


In [ ]:
kpis = {
    "sessions": session["session_id"].nunique(),
    "users": session["customer_id"].nunique(),
    "conversions": int(session["converted"].sum()),
    "conversion_rate_pct": round(session["converted"].mean()*100, 2),
    "avg_session_duration_seconds": round(session["session_duration_seconds"].mean(), 2),
    "avg_events_per_session": round(session["events_per_session"].mean(), 2)
}
kpis


## 7. Conversion by device/channel/product


In [ ]:
for col in ["device_type", "channel", "product"]:
    print("\n", col)
    print((session.groupby(col)
           .agg(sessions=("session_id","count"),
                conversions=("converted","sum"),
                conversion_rate=("converted","mean"))
           .assign(conversion_rate=lambda x: (x["conversion_rate"]*100).round(2))
           .sort_values("conversion_rate", ascending=False)))


## 8. Hypothesis test: device type vs conversion

H0: conversion is independent of device type.  
H1: conversion is associated with device type.


In [ ]:
contingency = pd.crosstab(session["device_type"], session["converted"])
chi2, p_value, dof, expected = chi2_contingency(contingency)
print(contingency)
print("Chi-square:", round(chi2, 4))
print("p-value:", p_value)


## 9. Machine-learning dataset


In [ ]:
features = [
    "device_type", "channel", "product", "city",
    "events_per_session", "session_duration_seconds"
]
X = session[features]
y = session["converted"]

cat_cols = ["device_type", "channel", "product", "city"]
num_cols = ["events_per_session", "session_duration_seconds"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


## 10. Logistic Regression


In [ ]:
logit = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])
logit.fit(X_train, y_train)
pred = logit.predict(X_test)
proba = logit.predict_proba(X_test)[:,1]
print(classification_report(y_test, pred))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))


## 11. Random Forest


In [ ]:
rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=250, random_state=42, class_weight="balanced"
    ))
])
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:,1]
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_proba), 4))


## 12. Gradient Boosting


In [ ]:
gb = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_proba = gb.predict_proba(X_test)[:,1]
print(classification_report(y_test, gb_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, gb_proba), 4))


## 13. Model comparison


In [ ]:
model_results = pd.DataFrame({
    "model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "roc_auc": [
        roc_auc_score(y_test, proba),
        roc_auc_score(y_test, rf_proba),
        roc_auc_score(y_test, gb_proba)
    ]
}).sort_values("roc_auc", ascending=False)
model_results


## 14. Business interpretation

Interpret the funnel, statistical test, segment conversion rates, and model results together.  
Avoid causal claims from observational clickstream data. Focus recommendations on measurable journey friction, testable hypotheses, and opportunities for controlled experimentation.
